# ReST - Vision Transformers

The same pipeline over the transformer hub (8 self-supervised + 8 supervised).
Self-supervised backbones have no classifier head, so their classifier weight is unavailable and
`G` falls back to the penultimate weight alone. Set `use_clf: false` in the config to drop the
classifier terms for every model.

Configuration lives in `configs/vit.yaml`.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd

from rest.config import load_config, record_path
from rest.data import set_seed
from rest.extract import calculate_transferability_scores
from rest.ground_truth import get_ground_truth
from rest.score import build_elements, rest_score, evaluate, weighted_kendall

## 1-2. Extract the stable-rank records

In [ ]:
cfg = load_config("../configs/vit.yaml")
set_seed(cfg.seed)
device = cfg.resolved_device()
ground_truth = get_ground_truth("vit")
print("device:", device)

all_json = {}
for dataset_name in [cfg.source_dataset] + cfg.target_datasets:
    path = record_path(cfg.out_dir, dataset_name)
    if os.path.exists(path):
        all_json[dataset_name] = json.load(open(path))
        print(f"=== {dataset_name}: cached ({len(all_json[dataset_name])} models)")
        continue

    print(f"\n=== {dataset_name} ===")
    records = {}
    for model_name in cfg.model_hub:
        try:
            record = calculate_transferability_scores(
                model_name, dataset_name,
                num_samples=cfg.num_samples, sample_seed=cfg.sample_seed,
                device=device, batch_size=cfg.batch_size, max_feats=cfg.max_feats)
        except Exception as exc:
            print(f"  [skip] {model_name}: {exc}")
            continue
        if record is None:
            continue
        record["finetune_accuracy"] = ground_truth.get(dataset_name, {}).get(model_name)
        records[model_name] = record
        print(f"  {model_name:<14} pen_sr={record['penultimate_layer'][0]:.3f} "
              f"clf_sr={record['classifier_layer'][0]:.3f}")

    all_json[dataset_name] = records
    json.dump(records, open(path, "w"), indent=2)

## 3. The four ReST elements

In [ ]:
df = build_elements(all_json, cfg.source_dataset, cfg.target_datasets,
                    get_ground_truth("vit"), use_clf=cfg.use_clf)
df.head(12)

## 4. ReST score

In [ ]:
df = rest_score(df, gamma=cfg.gamma)
df[["target dataset", "pre-trained model", "ReST", "fine-tune accuracy"]]

## 5. Weighted Kendall correlation, per group

In [ ]:
SSL_MODELS = ["mae_vitb16", "mae_vitl16", "dino_vitb16", "dino_vits8",
              "mocov3_vitb16", "simmim_vitb16", "mocov3_vits16", "dino_vits16"]
SUPERVISED_MODELS = ["vit_t_16", "vit_s_16", "vit_b_16", "pvtv2_b2",
                     "pvt_t", "pvt_s", "pvt_m", "swin_t"]

def eval_group(models, label):
    sub = df[df["pre-trained model"].isin(models)]
    taus = evaluate(sub, cfg.target_datasets)
    print(f"\n{label} (gamma={cfg.gamma}, use_clf={cfg.use_clf})")
    for dataset_name in cfg.target_datasets:
        print(f"  {dataset_name:<14}{taus[dataset_name]:>8.4f}")
    print(f"  {'MEAN':<14}{taus['MEAN']:>8.4f}")

eval_group(SSL_MODELS, "Self-supervised")
eval_group(SUPERVISED_MODELS, "Supervised")
eval_group(cfg.model_hub, "Combined")